In [ ]:
import models.utils as utils 
import models.agents.uni_random as uni_random
import models.agents.heuristics as heuristics
import models.agents.mcts as mcts
import analysis.analysis_utils as analysis_utils
import matplotlib.pylab as plt 
import seaborn as sns 
import itertools 
import pandas as pd
from tqdm import tqdm
from tqdm import trange
import random
import multiprocessing
import models.just_think.run_models 

import datetime
import importlib 
from itertools import chain
import constants
import numpy as np
import json

%matplotlib inline

In [ ]:
""" 
Init constants
"""
importlib.reload(constants)
model2name = constants.MODEL2NAME
model2pth = constants.MODEL2PTH["think"]

ordered_models = ["ours", "random",]

"""
Load in games, and filter to include just the games from the live gameplay study
"""
games, idx2game, game2idx, game_stimuli = analysis_utils.process_game_stimuli(constants.THINK_STIMULI_PTH)
livegameplay_games = pd.read_csv(constants.PLAY_STIMULI_PTH)
livegameplay_games["stimuli_id"] = [analysis_utils.get_stimuli_id(row) for idx, row in livegameplay_games.iterrows()]
play_game_objs = []
played_game_ids = set()
for game_id in livegameplay_games["stimuli_id"]:
    game = game_stimuli[game_id]
    played_game_ids.add(game["index"])
    game["stimuli_id"] = game_id
    play_game_objs.append(game)

"""
Load in human data
"""

with open(f"../human-data/play-exp/human-v-human/final-play/final_agg.json", "r") as f: 
    human_gameplay_data = json.load(f)  
        

think_human_df = pd.read_csv(constants.THINK_HUMAN_DATA)
# pulling out games according to Ced's game categories
all_game_types = {}
for i, entry in think_human_df.iterrows(): 
    game_type = entry.game_types
    game_id = entry.game_id
    if game_type not in all_game_types: all_game_types[game_type] = {game_id}
    else: all_game_types[game_type].add(game_id)

"""
Load in model preds
"""

model2res = {model: [] for model in ordered_models}
for model, model_pth in model2pth.items(): 
    res = analysis_utils.process_model_runs(model, model_pth, idx2game)
    if res is not None: model2res[model] = res


In [ ]:
all_humans = set()
all_arenas = set()

humans_per_game = {stim: set() for stim in human_gameplay_data} 
arena2players = {}
outcomes_per_game = {}
player2outcomes = {}
player2move_times = {}

show_times = False
overcame_advantage = set()

if show_times: 
    import math 
    n_game_show_rows = 6 
    n_game_show_cols = math.ceil(len(human_gameplay_data) / n_game_show_rows)
    fig, axes=  plt.subplots(n_game_show_rows, n_game_show_cols, figsize=(32,16))#, sharex=True)
    axes = axes.flatten()
    ax_idx = 0

for stim, entry in human_gameplay_data.items():
    stim_participants = {val for x in entry['order2player'] for val in x.values()}
    humans_per_game[stim] = stim_participants
    all_humans.update(stim_participants)
    all_arenas.update({a for a in entry['arena']})
    
    
    all_move_times = entry['move_times']
    
    # Find the maximum number of moves
    max_moves = max(len(game) for game in all_move_times)

    # Initialize arrays for means and standard errors
    means = []
    std_errors = []
    move_numbers = []
    game_len_avg = np.mean([len(eval(boards)) - 1 for boards in entry['boards']]) # rem empty board [check]

    # Calculate mean and standard error for each move (skipping first nan)
    for move in range(1, max_moves):
        # Get all values for this move number (excluding nan)
        move_times = [game[move]  for game in all_move_times if len(game) > move and not np.isnan(game[move])]
        if move_times:
            mean_time = np.mean(move_times)
            std_error = analysis_utils.compute_se(move_times) #np.std(move_times) / np.sqrt(len(move_times))
            means.append(mean_time)
            std_errors.append(std_error)
            move_numbers.append(move)

    if show_times: 

        ax = axes[ax_idx]
        ax.errorbar(move_numbers, means, yerr=std_errors, fmt='o-', capsize=5, 
                    color='blue', ecolor='gray', markersize=6, linewidth=2)
        ax.axvline(x=game_len_avg, color='r', linestyle='--')

        ax.set_xlabel('Move Number')
        ax.set_ylabel('Time (seconds)')
        n_rows_game, n_cols_game, win_conds = stim.split("*")
        n_rows_game = int(float(n_rows_game))
        n_cols_game = int(float(n_cols_game))
        n_cells = n_rows_game * n_cols_game
        
        game_title_fmt = f"{n_rows_game} x {n_cols_game}\n{analysis_utils.tidy_game_name(win_conds)}"
        ax.set_title(game_title_fmt)
       
        ax_idx += 1
    
    outcomes = []
    for payoff_data, order2player, outcome, arena, move_times in zip(entry["player_exp_payoffs"], entry["order2player"], entry["outcome"], 
                                                                     entry['arena'], entry['move_times']): 
        
        player2order = {v: k for k,v in order2player.items()}
        game_players = list(player2order)
        if len(game_players) > 0: arena2players.setdefault(arena, set())

        if len(payoff_data) != 0: 
            # NOTE: in pilots, some didn't have payoff data
            for player in payoff_data: 
                payoff = payoff_data[player]
                order = player2order[player]
                arena2players[arena].add(player)
        else: 
            for player in player2order: 
                arena2players[arena].add(player)
            
        outcome_int = 0  # 0, 1, -1
        if outcome == "Draw":
            outcome_int = 0 
            outcomes.append(outcome_int)
            
        else: 
            outcome_int = int(player2order[outcome]) # winner
            if outcome_int == 2: 
                outcome_int = -1 # second player
            outcomes.append(outcome_int)
            
        for player in game_players: 
            player2outcomes.setdefault(player, [])
            player_order = player2order[player]
            if outcome_int == 0: player_outcome = 0
            else: 
                if outcome == player: player_outcome = 1
                else: player_outcome = -1
            player2outcomes[player].append([player_outcome, stim, player_order])
           
        move_time_vals = move_times[1:] # note: nan is just the empty board at the start (no time) 
        if arena == '01JDQDCK4J093PE7H9JEWZF57X': 
            print(move_times)
            
            count_long = 0
            for i, m in enumerate(move_time_vals): 
                if m > means[i]: count_long += 1
            print("num long: ", count_long/len(move_time_vals))
            print(player2order, stim)
            
        for player, order in player2order.items():
            player2move_times.setdefault(player, [])
            if order == '1':
            # Player X makes the first, third, fifth... moves (even indices in 0-based indexing)
                move_times_player = move_time_vals[::2]
            elif order == '2':
                # Player O makes the second, fourth, sixth... moves (odd indices in 0-based indexing)
                move_times_player = move_time_vals[1::2]
            
            player2move_times[player].append(move_times_player)
                 
            
        
    
    outcomes_per_game[stim] = outcomes
    
if show_times: 
    axes[-1].set_xticks([])
    axes[-1].set_yticks([])
    axes[-1].spines['top'].set_visible(False)
    axes[-1].spines['right'].set_visible(False)
    axes[-1].spines['bottom'].set_visible(False)
    axes[-1].spines['left'].set_visible(False)
    plt.tight_layout()
    plt.savefig(save_dir + "move_times_grid.pdf", dpi=400)
    


In [ ]:
paramfit_dir = "../model-data/play-exp/final_data/ablated_simulations/"
import os
import pickle

all_param_files = [f for f in os.listdir(paramfit_dir) if '.DS_Store' not in f]
import ast 

process_pf2raw_pf = {}
for raw_pf in all_param_files: 
    raw_pf_dict = raw_pf.split(".pickle")[0]
    #params = {k: round(float(v), 1) for k, v in ast.literal_eval(raw_pf_dict).items()}
    params = raw_pf_dict
    process_pf2raw_pf[str(params)+".pickle"] = raw_pf

# make sure to include baseline and ablated, then sample others for now

''' 
Consider ablations of each, keeping the other fixed at 1.0
Three-way heatmap, averaging over one of the attributes
"Slices" 
'''
vals = np.arange(0, 2.2, 0.2)
attrs = ['center_weight', 'connect_weight', 'block_weight', ]
attr2viz = {'center_weight': 'Center Weight', 
            'block_weight': 'Block Weight', 
            'connect_weight': 'Connect Weight'}

default_attr_map = {attr: 1.0 for attr in attrs}
incl_params = [str(default_attr_map)]
from copy import copy


# Single attribute ablations (each attribute at 0, others at 1)
for attr in attrs:
    ablated_map = copy(default_attr_map)
    ablated_map[attr] = 0.0
    incl_params.append(str(ablated_map))

# Double attribute ablations (two at 0, one at 1)
for i, attr1 in enumerate(attrs):
    for attr2 in attrs[i+1:]:
        ablated_map = copy(default_attr_map)
        ablated_map[attr1] = 0.0
        ablated_map[attr2] = 0.0
        incl_params.append(str(ablated_map))
        

param_filenames = [process_pf2raw_pf[param + ".pickle"] for param in incl_params]

print(len(param_filenames))

In [ ]:
param_filenames

In [ ]:
all_files = [f for f in os.listdir(paramfit_dir) if f.endswith(".pickle")]
all_files

In [ ]:
import os
import ast

def load_paramfit_res(filename):
    param_setting_str = filename.split(".pickle")[0]
    try:
        param_setting = eval(param_setting_str)
    except Exception as e:
        print(f"Skipping {filename} due to param_setting eval error: {e}")
        return None

    filepath = os.path.join(paramfit_dir, filename)
    try:
        with open(filepath, "rb") as f:
            res = pickle.load(f)
    except Exception as e:
        print(f"Skipping {filename} due to content eval error: {e}")
        return None

    return res, param_setting

# Filter only .pickle files
all_files = [f for f in os.listdir(paramfit_dir) if f.endswith(".pickle")]

param_res = [r for f in all_files if (r := load_paramfit_res(f)) is not None]

# Make a dict of the results
model_fit_data = {str(paramname): res for res, paramname in param_res}
all_agents = list(model_fit_data.keys())


In [ ]:

'''
Process the human watch data
''' 
with open("../human-data/watch-exp/final_res.json", "r") as f: watch_data = json.load(f)


# rem IDs that used standard values too much [as pre-reg]

rem_ids = ['MUJ2ELF5D6',
 'XAOHNXHKQL',
 'FRF6G0BXYN',
 'J090IDCESB',
 'ZMSROEENK1',
 'ZRSSZKF1VU',
 'B8RXDIXOCL',
 'M9SW86IBCL',
 '2WKOW6RBSH',
 'O5CGUJIT5A']
''' 
Extract key info from the original human game 

{game: {arena: { 
}
}
'''

game_stages = ['early', 'intermediate', 'late']
order2stage = {i: v for i, v in enumerate(game_stages)}

move_idx_offset = -1

played_game_data = {}

for uid, uid_data in watch_data.items(): 
    if uid in rem_ids: continue
    for game, game_res in uid_data.items(): 
        batch_metadata = game_res['batch_metadata']
        arena = batch_metadata['arena'] 
        
        played_moves = batch_metadata['played_pos']
        
        if game not in played_game_data: played_game_data[game] = {}
        if arena not in played_game_data[game]: played_game_data[game][arena] = {}
        
        # get moves 'so far' for each board 
        boards = batch_metadata['boards']
        processed_boards = analysis_utils.process_game_states(boards)
        pred_move_idxs = batch_metadata['pred_move_idxs']
        
        # get game outcome 
        play_data = human_gameplay_data[game]
        player2order = None
        # make sure to pull out the "right" arena
        for arena2, order2player, outcome, all_move_times in zip(play_data['arena'],  
                                                 play_data["order2player"], 
                                                 play_data["outcome"],
                                                 play_data['move_times']): 
            if arena != arena2: continue
            player2order = {v: k for k,v in order2player.items()}

            if outcome == "Draw": 
                obs_outcome = 0 
            else: 
                obs_outcome = int(player2order[outcome])
                
            played_move_times = [all_move_times[idx] for idx in pred_move_idxs]
                
            break
        
        boards_at_stage = {}
        
        for order, move_idx in enumerate(pred_move_idxs): 
            stage = game_stages[order]
            boards_at_stage[stage] = processed_boards[move_idx + move_idx_offset]
            
        played_game_data[game][arena] = {
            'boards_at_stage': boards_at_stage,
            'played_pos': played_moves,
            'obs_outcome': obs_outcome,
            'move_idxs': pred_move_idxs, 
            'player2order': player2order, 
            'played_move_times': played_move_times,
            'player_at_pos': batch_metadata['player_at_idx']
            
        }
        
all_games = sorted(played_game_data.keys())
                

human_watch_data = {game: {} for game in all_games}
human_watch_payoffs = {game: [] for game in all_games}
human_watch_fun = {game: [] for game in all_games}

for user, user_data in watch_data.items(): 

    for game, game_data in user_data.items(): 
        arena = game_data['batch_metadata']['arena']
        
        if arena not in human_watch_data[game]: 
            human_watch_data[game][arena] = {game_stage: {
                'dist': [], 'rt': [], 'uid': []
                } for game_stage in game_stages}
            
        payoff = game_data['judgements'][-1]
        if payoff is not None: 
            human_watch_payoffs[game].append(payoff)
        else: 
            if 'response' in game_data['judgements'][0]: human_watch_fun[game].append(game_data['judgements'][0]['response'])
            
        for order, entry in enumerate(game_data['pred_moves']):
            raw_pred_moves, _, _, _, rt = entry
            game_stage = game_stages[order]
            
            # convert click count format of keys: 'row-col' --> (row, col)
            pred_move_dist = {}
            n_rows_game, n_cols_game, win_conds = game.split("*")
            n_rows_game = int(float(n_rows_game))
            n_cols_game = int(float(n_cols_game))
            
            for r in range(n_rows_game): 
                for c in range(n_cols_game): 
                    pred_move_dist[(r, c)] = 0
            norm_counts = int(np.sum(list(raw_pred_moves.values())))
            for k, v in raw_pred_moves.items(): 
                row, col = k.split('-')
                row = int(row)
                col = int(col)
                k = (row,col)
                pred_move_dist[k] = v / norm_counts

            human_watch_data[game][arena][game_stage]['dist'].append(pred_move_dist)
            human_watch_data[game][arena][game_stage]['rt'].append(rt)
            human_watch_data[game][arena][game_stage]['uid'].append(user)



In [ ]:
len(human_gameplay_data)

In [ ]:
len(played_game_data.keys())

In [ ]:

''' 
Process the model data 
'''
agents = all_agents
model_watch_data = {agent: 
    {game: {} for game in played_game_data.keys()}
    for agent in agents
    }

for agent in agents: 
    for game in played_game_data.keys(): 
        game_id = game2idx[game]
        game_obj = games[game_id - 1]
        
        for arena, arena_data in played_game_data[game].items(): 
            model_watch_data[agent][game][arena] = {game_stage: [] for game_stage in game_stages}
            
            boards = arena_data['boards_at_stage']
            
            move_idxs = arena_data['move_idxs']
            
            for order, game_stage in enumerate(game_stages): 
                
                move_idx = move_idxs[order] + move_idx_offset
                board = boards[game_stage]
                # just take the plays 
                board = [[piece for piece, move_turn in entries] for entries in board]
                board = analysis_utils.convert_board_repr(board)
                    
                if agent != "random": 
                    try: 
                        raw_m_outputs = model_fit_data[agent][arena][str(game_obj['index'])][str(move_idx)]['dist']
                    
                        if type(raw_m_outputs) != list: raw_m_outputs =[raw_m_outputs]
                        
                        m_outputs = []
                        for m_output in raw_m_outputs: 
                            new_dict = {}
                            for k, v in m_output.items():
                                if v != -np.inf: new_dict[k] = v
                            m_outputs.append(new_dict)
                        
                        
                        m_dist = [utils.softmax_dist(m_output, T=1) for m_output in m_outputs]
                    except: 
                        m_dist = None
                        #continue 
                else: 
                    
                    # get all possible legal moves, and use this to ground comparisons
                    m_dist = [uniform_dist(board)]
                    
                model_watch_data[agent][game][arena][game_stage] = m_dist
            

In [ ]:
''' 
Three-way heatmap, averaging over one of the attributes
"Slices" 
'''
vals = np.arange(0, 2.2, 0.2)
attrs = ['center_weight','connect_weight', 'block_weight', ]
attr2viz = {'center_weight': 'Center Weight', 
            'block_weight': 'Block Weight', 
            'connect_weight': 'Connect Weight'}

default_attr_map = {attr: 1.0 for attr in attrs}

In [ ]:
vals = np.arange(0, 2.2, 0.2)
attrs = ['center_weight', 'connect_weight', 'block_weight', ]
attr2viz = {'center_weight': 'Center Weight', 
            'block_weight': 'Block Weight', 
            'connect_weight': 'Connect Weight'}

default_attr_map = {attr: 1.0 for attr in attrs}
incl_params = [str(default_attr_map)]
from copy import copy


# Single attribute ablations (each attribute at 0, others at 1)
for attr in attrs:
    ablated_map = copy(default_attr_map)
    ablated_map[attr] = 0.0
    incl_params.append(str(ablated_map))

# Double attribute ablations (two at 0, one at 1)
for i, attr1 in enumerate(attrs):
    for attr2 in attrs[i+1:]:
        ablated_map = copy(default_attr_map)
        ablated_map[attr1] = 0.0
        ablated_map[attr2] = 0.0
        incl_params.append(str(ablated_map))

ablation_filenames = [param for param in incl_params]

In [ ]:
''' 
Old format: 
- all_participant_data = {arena: {game: score, game: score}}
- all_model_data = {game: dist}

New: 
- all_participant_data = {game: {arena: {move_idx: move}}}
- all_model_data = {game: {arena: {move_idx: dist}}}

'''
importlib.reload(models.just_think.run_models)
importlib.reload(analysis_utils)
from models.agents.uni_random import uniform_dist
considered_games = sorted(list(human_gameplay_data))
all_participant_data = {stim: {} for stim in considered_games}

all_agents = ablation_filenames
all_model_data = {agent: {stim: {} for stim in considered_games} for agent in all_agents}


from scipy.stats import entropy
importlib.reload(models.just_think.run_models)
importlib.reload(analysis_utils)
from models.agents.uni_random import uniform_dist
considered_games = sorted(list(human_gameplay_data))
all_participant_data = {stim: {} for stim in considered_games}

all_model_data = {agent: {stim: {} for stim in considered_games} for agent in all_agents}

# map the game, arena, and move symbol to a particular player
participant_id_map = {}
participant2arena = {}
skip_arenas =set()
for agent in all_agents: 

    for idx, (stim, stim_data) in  enumerate(human_gameplay_data.items()): 
        game_id = game2idx[stim]
        game = games[game_id - 1]
        all_game_boards = [eval(board) for board in stim_data["boards"]]
        
        n_cols = len(all_game_boards[0])
        n_rows = len(all_game_boards[0][0])
        n_cells = n_cols * n_rows
        
        for arena_idx, game_boards in enumerate(all_game_boards): 
            arena_name = stim_data["arena"][arena_idx]
            
            if arena_name in skip_arenas: continue
            order2player = stim_data['order2player'][arena_idx]
            player2order = {player: 'X' if order == '1' else 'O' for order, player in order2player.items()}
            
            for player, order in player2order.items(): 
                if player not in participant_id_map: 
                    participant_id_map[player] = {stim: order, "arena": arena_name}
                else: participant_id_map[player][stim] = order
            
            if len(game_boards) == 0: continue
            starting_board = analysis_utils.convert_board_repr(game_boards[0])
            prev_board = starting_board
            
            arena_data_h = {}
            arena_data_m = {}
                    
            for i, board in enumerate(game_boards[1:]): 

                if agent == "random":
                    move_dist = [uniform_dist(prev_board)]

                else: 
                    try:
                        game_output = model_fit_data[agent][arena_name][str(game['index'])]
                        output = game_output[str(i)]
                        # just one dist for now 
                        move_dist = output['dist']
                        if move_dist is None: 
                            print("NONE: ", agent, i, stim, arena)
                        
                        if type(move_dist) != list:
                            
                            move_dist = [move_dist] 
                        move_dist = [move_dist[0]]
                        
                        m_outputs = []
                        for m_output in move_dist: 
                            new_dict = {}
                            for k, v in m_output.items():
                                if v != -np.inf: new_dict[k] = v
                            m_outputs.append(new_dict)
                        
                        move_dist = m_outputs
                        
                        
                    except Exception as e:
                        print("error: ", e, agent, stim, arena_name,
                              "\n\t Move: ", str(i+1), " Game: ", str(game['index']), 
                              model_fit_data[agent][arena_name].keys(),
                              "\n\t Move keys: ", game_output.keys()) 
                        continue 
                
                
                round_board = analysis_utils.convert_board_repr(board)
                # what was this player's next move
                diff = np.array(round_board) != np.array(prev_board)  # Find where the boards differ
                row, col = np.where(diff)  # Get the indices of the differences
                # just one move
                move = (int(row[0]), int(col[0]))
                player_id = round_board[move[0]][move[1]]
                num_legal = np.sum(np.array(prev_board) == '_')
                
                arena_data_h[i] = (move, player_id)
                arena_data_m[i] = (move_dist, num_legal)
                
                
                
                prev_board = round_board
                
                
            all_participant_data[stim][arena_name] = arena_data_h
            all_model_data[agent][stim][arena_name] = arena_data_m



                        


In [ ]:
agent

In [ ]:
all_model_data.keys()

In [ ]:
vals = all_model_data["{'center_weight': 1.0, 'connect_weight': 1.0, 'block_weight': 1.0}"]['1.0*10.0*3 pieces in a row wins.']['01JNGTBK5X351NZBEGZFZ18KJB'][0]#.keys()

len(vals[0])

In [ ]:
''' 
Vary slop params
'''
import admixture_utils as pf_utils
import gc
import random
importlib.reload(pf_utils)
np.random.seed(7)
random.seed(7)
step = 0.05
max_alpha_val = 1.0-(1e-10)
alpha_vals= list(np.arange(0.5, max_alpha_val, step))

arena_subset_prop = -1
game_subset_prop = -1
n_bootstrap_samples = 1
fixed_alpha_res = {}
game_stage = 'all'
viz_agents = all_agents
for agent in viz_agents:
    fixed_alpha_res.setdefault(agent, {alpha: [] for alpha in alpha_vals})
    for alpha in alpha_vals: 
        for j in range(n_bootstrap_samples): 
            print("on trial: ", j, alpha, agent)
            score, all_scores, _, game_ordering = pf_utils.score_fixed_alpha(agent,
                                                                                         all_model_data, 
                                                                                         all_participant_data, 
                                                                                         alpha=alpha,
                            arena_subset_prop=arena_subset_prop, game_subset_prop=game_subset_prop,
                            game_stage=game_stage, temperature = 1)
            fixed_alpha_res[agent][alpha].append([all_scores,game_ordering])
            print("done run")
            gc.collect()

agg_scores_alpha= {}
for agent_idx, agent in enumerate(viz_agents): 
    collapsed_scores = []
    for alpha in alpha_vals: 
        collapsed_scores.extend([x for x in fixed_alpha_res[agent][alpha] if 'inf' not in str(x)])
    agg_scores_alpha[agent] = collapsed_scores



In [ ]:
all_model_data.keys()

In [ ]:
with open("play_paramfit.json", "w") as f: 
    json.dump({'agg': agg_scores_alpha, 'individ': fixed_alpha_res}, f)

In [ ]:
param2bootstrap_res = {agent: [] for agent in all_agents}
for agent, collapsed_scores in agg_scores_alpha.items():#.iterrows():
    #score = np.mean(collapsed_scores)
    # agent = entry.agent
    param2bootstrap_res[agent].extend(collapsed_scores)#score)

In [ ]:
individ_move_scores_alpha = {}
agg_scores_alpha = {}

for agent in viz_agents:
    # We'll flatten using sum(..., []) exactly like before
    move_scores = []
    for alpha in fixed_alpha_res[agent]:
        for score_bundle in fixed_alpha_res[agent][alpha]:
            all_scores, game_ordering = score_bundle
            # Flatten all_scores if it's a list of lists
            if isinstance(all_scores, list) and isinstance(all_scores[0], list):
                flat_scores = sum(all_scores, [])
            else:
                flat_scores = all_scores  # Already flat
            move_scores.extend(flat_scores)
    agg_scores_alpha[agent] = move_scores
    individ_move_scores_alpha[agent] = np.array(move_scores)


In [ ]:

''' 
Process the model data 
'''

all_agents = ablation_filenames
agents = all_agents
model_watch_data = {agent: 
    {game: {} for game in played_game_data.keys()}
    for agent in agents
    }

for agent in agents: 
    for game in played_game_data.keys(): 
        game_id = game2idx[game]
        game_obj = games[game_id - 1]
        
        for arena, arena_data in played_game_data[game].items(): 
            model_watch_data[agent][game][arena] = {game_stage: [] for game_stage in game_stages}
            
            boards = arena_data['boards_at_stage']
            
            move_idxs = arena_data['move_idxs']
            
            for order, game_stage in enumerate(game_stages): 
                
                move_idx = move_idxs[order] + move_idx_offset
                board = boards[game_stage]
                # just take the plays 
                board = [[piece for piece, move_turn in entries] for entries in board]
                board = analysis_utils.convert_board_repr(board)
                    
                if agent != "random": 
                    try: 
                        raw_m_outputs = model_fit_data[agent][arena][str(game_obj['index'])][str(move_idx)]['dist']
                    
                        if type(raw_m_outputs) != list: raw_m_outputs =[raw_m_outputs]
                        
                        # convert any -inf to 0 --- check?? or drop key?
                        m_outputs = []
                        for m_output in raw_m_outputs: 
                            new_dict = {}
                            for k, v in m_output.items():
                                if v != -np.inf: new_dict[k] = v
                            m_outputs.append(new_dict)
                        
                        
                        m_dist = [utils.softmax_dist(m_output, T=1) for m_output in m_outputs]
                    except Exception as e: 
                        print("missing: ", e, model_fit_data[agent][arena][str(game_obj['index'])].keys())
                        m_dist = None
                        #continue 
                else: 
                    
                    # get all possible legal moves, and use this to ground comparisons
                    m_dist = [uniform_dist(board)]
                    
                model_watch_data[agent][game][arena][game_stage] = m_dist
            

In [ ]:
model_watch_data.keys()

In [ ]:

importlib.reload(analysis_utils)
metrics = [
    'TV',
          # 'JSD'
           #'corr'
           ]



metric2func = {'KL': analysis_utils.get_kl,
                  'TV': analysis_utils.get_tv,
                  }#'Wasserstein': analysis_utils.get_wasserstein}
n_samples_per_move = 10

metric_data = {
    'game': [],
    'arena': [],
    'stage': [],
    'agent': [],
    'human_data_type': [],
    'alpha': [],
    'Configuration': []
}
# agents = all_agents#sub_agents #all_agents
metric_data.update({metric: [] for metric in metrics})

config_dict2name = {
"{'center_weight': 1.0, 'connect_weight': 1.0, 'block_weight': 1.0}": "All Components",
 "{'center_weight': 0.0, 'connect_weight': 1.0, 'block_weight': 1.0}": "Ablate Center Weight",
 "{'center_weight': 1.0, 'connect_weight': 0.0, 'block_weight': 1.0}": 'Ablate Connect Weight',
 "{'center_weight': 1.0, 'connect_weight': 1.0, 'block_weight': 0.0}": 'Ablate Block Weight',
 "{'center_weight': 0.0, 'connect_weight': 0.0, 'block_weight': 1.0}": 'Only Block Weight',
 "{'center_weight': 0.0, 'connect_weight': 1.0, 'block_weight': 0.0}": 'Only Connect Weight',
 "{'center_weight': 1.0, 'connect_weight': 0.0, 'block_weight': 0.0}": 'Only Center Weight',
    
}



np.random.seed(7)
random.seed(7)

agg_human_dists_watch ={}

step = 0.1

alpha_vals= list(np.arange(0.5, 1.0, step))
alpha_vals.append(1.0-(1e-10))
agg_human= True

agent2corrs = {agent: {} for agent in agents}

for alpha in alpha_vals: 
    agent2lists ={agent: [] for agent in agents}
    for game, game_data in played_game_data.items():
        for arena, arena_data in game_data.items(): 
            # arena_data has data for all stages
            
            played_moves = arena_data['played_pos']
            
            for order, game_stage in enumerate(game_stages): 
                h_dists = human_watch_data[game][arena][game_stage]['dist']
                if agg_human: h_dists = [analysis_utils.average_dicts(h_dists)]


                


                draw_samples = False

                for agent in agents:  
                    
                    if agent == 'ours' and alpha == 0.5: 

                        agg_human_dists_watch.setdefault(game, {})
                        agg_human_dists_watch[game].setdefault(game_stage, {})
                        agg_human_dists_watch[game][game_stage].setdefault(arena, {})
                        agg_human_dists_watch[game][game_stage][arena] = h_dists[0]
                    
                    
                    m_dists = model_watch_data[agent][game][arena][game_stage]
                    if m_dists is None: 
                        print("missing: ", game, arena, game_stage)
                        continue
                    draw_samples = True
                    
            
                    # compute metrics 
                    scores = []
                    for metric_idx, metric in enumerate(metrics): 
                        
                        m_dist = m_dists[0]
                        h_dist = h_dists[0]
                        for k,vh in h_dist.items(): 
                            if k not in m_dist and vh != 0: print(k, vh)

                        if metric == 'KL':
                            scores = [np.mean([analysis_utils.get_kl(h_dist, m_dist, alpha=alpha) for m_dist in m_dists]) for h_dist in h_dists]
                        elif metric == 'TV':
                            scores = [np.mean([analysis_utils.get_tv(h_dist, m_dist, alpha=alpha) for m_dist in m_dists])  for h_dist in h_dists]
                        elif metric == 'corr':
                            scores = [np.mean([get_board_corr(h_dist, m_dist, alpha=alpha) for m_dist in m_dists])  for h_dist in h_dists]
                        elif metric == 'JSD': 
                            scores = [np.mean([analysis_utils.get_jsd(h_dist, m_dist, alpha=alpha) for m_dist in m_dists])  for h_dist in h_dists]

                        
                        if metric_idx == 0: 
                            metric_data['game'].extend([game for _ in range(len(h_dists))])
                            metric_data['arena'].extend([arena for _ in range(len(h_dists))])
                            metric_data['agent'].extend([agent for _ in range(len(h_dists))])
                            metric_data['stage'].extend([game_stage for _ in range(len(h_dists))])
                            metric_data['human_data_type'].extend(['should' for _ in range(len(h_dists))])
                            metric_data['alpha'].extend([alpha for _ in range(len(h_dists))])
                            metric_data['Configuration'].extend([config_dict2name[agent] for _ in range(len(h_dists))])
                            #metric_data['Configuration'].extend([agent for _ in range(len(h_dists))])
                        for score in scores:
                            metric_data[metric].append(score)


metric_df = pd.DataFrame.from_dict(metric_data)
metric_df.to_csv(f"final_processed_res/param_fits/watch_tvd.csv")

In [ ]:
metric_df


In [ ]:
ablation_filenames